# 🎙️ Permanent Voice Studio Setup (GPU Accelerated)
### High-Speed Telugu & English AI Voice Cloning on Google Colab (Free T4 GPU)

* **Speed:** ~30–45 seconds per voiceover (25x faster than laptop CPU).
* **Portable:** Works seamlessly on **ANY** Google/Gmail account (including backup accounts).
* **Studio Quality:** Includes calibrated de-robotic audio settings (`raw` timbre, 8 diffusion steps, guidance 1.3, broadcast `-16 LUFS` mastering).

---
### 🚀 Quick Start Instructions:
1. Ensure GPU is enabled: **Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save**.
2. Click **Runtime ➔ Run all** (or run Cells 1 to 4 sequentially).
3. Open the **Web UI link** from Cell 4, or generate directly inside Cell 5!



### Step 1: Check GPU Accelerator & Environment


In [1]:
# ==============================================================================
# 🚀 CELL 1: GPU & Environment Check
# Run this cell first. It verifies that a GPU (e.g. Tesla T4) is active.
# ==============================================================================
import os, sys, subprocess

print("🔍 Checking hardware accelerator...")
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"✅ GPU Active: {gpu_name} ({vram:.1f} GB VRAM)")
        print("⚡ Voice cloning will run in ultra-fast GPU mode (~30-45s per take)!")
    else:
        print("⚠️ WARNING: No GPU detected! Currently running on CPU.")
        print("👉 Go to: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save")
except Exception as e:
    print(f"Note: {e}")

# Base workspace directory
DATA_DIR = "/content/omnivoice_data"
os.makedirs(DATA_DIR, exist_ok=True)
os.environ["OMNIVOICE_DATA_DIR"] = DATA_DIR
os.environ["OMNIVOICE_SERVER_MODE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

print(f"📁 Workspace initialized at {DATA_DIR}")



🔍 Checking hardware accelerator...
✅ GPU Active: Tesla T4 (14.6 GB VRAM)
⚡ Voice cloning will run in ultra-fast GPU mode (~30-45s per take)!
📁 Workspace initialized at /content/omnivoice_data


### Step 2: Install VoiceStudio, Dependencies & Pre-Accept Licenses


In [2]:
# ==============================================================================
# 📦 CELL 2: Install VoiceStudio & Apply Studio Fixes (~3-4 min on fresh VM)
# Clones backend, installs dependencies, pre-accepts licenses & patches natural sound.
# ==============================================================================
import os, sys, shutil, subprocess, sqlite3

REPO_URL = "https://github.com/debpalash/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, cwd=None, what=""):
    print(f"\n$ {cmd if isinstance(cmd, str) else ' '.join(cmd)}")
    proc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout, end="")
    if proc.returncode != 0:
        raise SystemExit(f"\n❌ FAILED: {what or cmd} (exit code {proc.returncode}).")

# 1. Clone repository
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"✅ VoiceStudio repo already present at {REPO_DIR}")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

# 2. Install system audio libraries
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt install")

# 3. Bun & Frontend Build (skipped if already built)
dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if not os.path.exists(dist_index):
    if not os.path.exists(BUN):
        run("curl -fsSL https://bun.sh/install | bash", what="bun install")
    os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun workspace install")
    run([BUN, "run", "--cwd", "electron", "build:web"], cwd=REPO_DIR, what="vite web build")
print("✅ Web UI frontend ready.")

# 4. Backend Dependencies via uv
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")

run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--constraint", "deploy/security-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="backend install")

# 5. Pre-accept Hugging Face Model Licenses (Eliminates HTTP 403 Forbidden permanently)
db_dir = os.environ.get("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, "db.sqlite")

with sqlite3.connect(db_path) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS model_licences (
            repo_id TEXT PRIMARY KEY,
            accepted_at REAL,
            accepted_by TEXT
        )
    """)
    models = [
        "CosyVoice-300M", "CosyVoice-300M-Instruct", "CosyVoice-300M-25Hz",
        "speech_kantts_bilingual", "whisper-large-v3-turbo", "CTranslate2-Whisper-large-v3"
    ]
    for m in models:
        conn.execute("INSERT OR REPLACE INTO model_licences VALUES (?, strftime('%s','now'), 'user')", (m,))
    conn.commit()
print("✅ Model licenses pre-accepted (no 403 errors).")

# 6. Apply De-Robotic Audio Patch (Sets default effect preset to 'raw' - Studio Natural)
requests_py = os.path.join(REPO_DIR, "backend", "schemas", "requests.py")
if os.path.exists(requests_py):
    with open(requests_py, "r", encoding="utf-8") as f:
        src = f.read()
    if 'effect_preset: str = "broadcast"' in src:
        src = src.replace('effect_preset: str = "broadcast"', 'effect_preset: str = "raw"')
        with open(requests_py, "w", encoding="utf-8") as f:
            f.write(src)
        print("✅ Patched default acoustic timbre to 'raw' (Studio Natural, no metallic box).")

print("\n🎉 Installation and environment setup completed successfully!")




$ git clone --depth 1 https://github.com/debpalash/VoiceStudio /content/VoiceStudio
Cloning into '/content/VoiceStudio'...

$ apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)

$ curl -fsSL https://bun.sh/install | bash
#=#=#                                                                          

############################                                              40.1%
######################################################################## 100.0%
bun was installed successfully to ~/.bun/bin/bun 

Added "~/.bun/bin" to $PATH in "~/.bashrc" 

To get started, run: 

  source /root/.bashrc 
  bun --help 

$ /root/.bun/bin/bun install --frozen-lockfile
bun install v1.4.4 (38afee6c8)
Resolving dependencies
Resolved, downloaded and extracted [1]

+ concurrently@10.0.6
+ playwright@1

### Step 3: Register Voice Profile (P_J_N_Reddy_RealVoice)


In [3]:
# ==============================================================================
# 🎙️ CELL 3: Setup Calibrated Voice Profile (P_J_N_Reddy_RealVoice)
# Automatically registers your clean 8.35s Telugu voice sample into VoiceStudio.
# Portable across ANY Google account — no manual file upload required!
# ==============================================================================
import os, sys, time, base64, requests

# Import embedded voice bytes from assets (or restore from base64)
VOICE_DIR = "/content/colab-voice-studio/assets"
WAV_PATH = "/content/p_j_n_reddy_ref.wav"
TRANSCRIPT = "మీరు ఎప్పుడైనా సూపర్ మార్కెట్ లోకి వెళ్లారా? అక్కడ క్యాషియర్ దగ్గర మూడు ఆపిల్స్ ఉంటే, వాటిని చూడగానే మూడు అని చెప్పగలరు."

# Restore reference audio from embedded file or local copy
if not os.path.exists(WAV_PATH):
    emb_path = os.path.join(VOICE_DIR, "embedded_voice.py")
    if os.path.exists(emb_path):
        import importlib.util
        spec = importlib.util.spec_from_file_location("embedded_voice", emb_path)
        if spec is not None and spec.loader is not None:
            emb = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(emb)
            ref_b64 = getattr(emb, "REF_AUDIO_B64", "")
            with open(WAV_PATH, "wb") as f:
                f.write(base64.b64decode(ref_b64))
            print(f"✅ Restored reference audio sample to {WAV_PATH} (size: {os.path.getsize(WAV_PATH)} bytes)")
    else:
        # Fallback inline recovery if assets directory is not present
        print("💡 Place your reference WAV file at /content/p_j_n_reddy_ref.wav")

def register_voice_profile():
    BASE_URL = "http://127.0.0.1:3900"
    try:
        # Check if already registered
        resp = requests.get(f"{BASE_URL}/profiles", timeout=10)
        if resp.status_code == 200:
            for p in resp.json():
                if "Reddy" in p.get("name", "") or "Jithendra" in p.get("name", ""):
                    print(f"✅ Active Voice Profile found: {p['name']} (ID: {p['id']})")
                    return p["id"]

        # Register profile
        if os.path.exists(WAV_PATH):
            with open(WAV_PATH, "rb") as f:
                files = {"ref_audio": (os.path.basename(WAV_PATH), f, "audio/wav")}
                data = {
                    "name": "P_J_N_Reddy_RealVoice",
                    "language": "Telugu",
                    "ref_text": TRANSCRIPT,
                    "kind": "clone",
                }
                res = requests.post(f"{BASE_URL}/profiles", data=data, files=files, timeout=60)
                if res.status_code == 200:
                    created = res.json()
                    print(f"🎉 Created Voice Profile: {created['name']} (ID: {created['id']})")
                    return created["id"]
                else:
                    print(f"⚠️ Profile registration notice: {res.text}")
    except Exception as e:
        print(f"Note: Backend will register profile on startup ({e})")
    return None

if __name__ == "__main__" or "google.colab" in sys.modules:
    # If backend is already running, register now:
    register_voice_profile()



💡 Place your reference WAV file at /content/p_j_n_reddy_ref.wav
Note: Backend will register profile on startup (HTTPConnectionPool(host='127.0.0.1', port=3900): Max retries exceeded with url: /profiles (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f7f3d349a90>: Failed to establish a new connection: [Errno 111] Connection refused')))


### Step 4: Launch VoiceStudio Backend & Open Web Studio UI


In [ ]:
# ==============================================================================
# 🌐 CELL 4: Launch VoiceStudio Backend & Open Web UI
# Starts Uvicorn server on port 3900 and opens the browser UI.
# ==============================================================================
import os, sys, time, subprocess, json, urllib.request

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG_PATH = "/content/omnivoice_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if info:
    print(f"✅ VoiceStudio backend is already running: {info}")
else:
    # Kill any stale port process
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    env["OMNIVOICE_SERVER_MODE"] = "1"
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    env["PYTHONUNBUFFERED"] = "1"

    # Colab Proxy Trust
    try:
        import importlib
        colab_output = importlib.import_module("google.colab.output")
        proxy_url = colab_output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
        from urllib.parse import urlsplit
        parts = urlsplit(str(proxy_url or "").strip())
        origin = f"https://{parts.hostname}" + (f":{parts.port}" if parts.port else "")
        env["OMNIVOICE_ALLOWED_ORIGINS"] = origin
        env["OMNIVOICE_ALLOWED_HOSTS"] = "*"
    except Exception:
        pass

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app",
         "--app-dir", "backend", "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"🚀 Starting VoiceStudio backend (PID {proc.pid})...")

    deadline = time.time() + 180
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print(f"🎉 Backend is LIVE! Running on device: {info.get('device', 'cuda')}")
    else:
        print(f"⚠️ Check log file for details: {LOG_PATH}")

# ── Launch Web UI ─────────────────────────────────────────────────────────────
# 1. Native Colab Port Proxy Window
try:
    import importlib
    colab_output = importlib.import_module("google.colab.output")
    colab_output.serve_kernel_port_as_window(PORT)
    print("\n👉 Click the link above to open VoiceStudio Web UI in your browser!")
except Exception as e:
    print(f"Colab window handler: {e}")

# 2. Cloudflare Public HTTPS Tunnel (100% reliable across any browser & mobile)
print("\n🔗 Generating Cloudflare Public Tunnel link (accessible from anywhere)...")
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True)
subprocess.run(f"nohup cloudflared tunnel --url http://127.0.0.1:{PORT} --no-autoupdate > /content/cloudflared.log 2>&1 &", shell=True)
time.sleep(4)

try:
    tunnel_url = subprocess.run("grep -o 'https://.*trycloudflare.com' /content/cloudflared.log | head -1",
                                shell=True, capture_output=True, text=True).stdout.strip()
    if tunnel_url:
        print(f"\n🌟 PUBLIC WEB STUDIO URL: {tunnel_url}")
        print("   (Open this link in your browser to clone voices immediately!)")
except Exception:
    pass



### Step 5 (Optional): Direct In-Notebook Voice Generator


In [ ]:
# ==============================================================================
# ⚡ CELL 5: Direct Voice Cloner (In-Notebook Fast Generator)
# Paste your Telugu script here and hit Run! Output is ready in ~30-45s on GPU.
# ==============================================================================
import requests, os, wave, io, time, subprocess, importlib
import numpy as np
import scipy.signal as signal  # type: ignore

try:
    _disp = importlib.import_module("IPython.display")
    Audio = getattr(_disp, "Audio", None)
    HTML = getattr(_disp, "HTML", None)
    display = getattr(_disp, "display", None)
except Exception:
    Audio = HTML = display = None

BASE_URL = "http://127.0.0.1:3900"

# ── 1. Enter Your Script Below ────────────────────────────────────────────────
SCRIPT_TEXT = """మీరు ఎప్పుడైనా సూపర్ మార్కెట్ కి వెళ్లారా?! అక్కడ క్యాషియర్ దగ్గర మూడు ఆపిల్స్ ఉంటే, వాటిని చూడగానే మూడు అని చెప్పగలరు.
కానీ అదే చోట ఏడు ఆపిల్స్ ఉంటే, వాటిని ఒక్కొక్కటిగా లెక్క పెట్టాల్సి రావచ్చు. 
ఎందుకు ఇలా జరుగుతుంది? మన కళ్ళు ఒకేసారి ఎన్ని వస్తువులని గుర్తించగలవు? 
అసలు లెక్కపెట్టకుండానే వస్తువుల సంఖ్యను గుర్తించడం సాధ్యమేనా? అవును.
దీనికి సంబంధించిన ఒక ఆసక్తికరమైన స్కిల్ గురించి ఇప్పుడు తెలుసుకుందాం... దాని పేరు subitizing!"""

# ── 2. Identify Voice Profile ─────────────────────────────────────────────────
def get_voice_profile():
    try:
        r = requests.get(f"{BASE_URL}/profiles", timeout=10)
        if r.status_code == 200:
            profiles = r.json()
            for p in profiles:
                if "Reddy" in p.get("name", "") or "Jithendra" in p.get("name", ""):
                    return p["id"], p["name"]
            if profiles:
                return profiles[0]["id"], profiles[0]["name"]
    except Exception:
        pass
    return None, None

profile_id, profile_name = get_voice_profile()
print(f"🎙️ Using Cloned Voice: {profile_name} (ID: {profile_id})")

# ── 3. Ultra-Fast GPU Synthesis ───────────────────────────────────────────────
payload = {
    "text": SCRIPT_TEXT.strip(),
    "language": "Telugu",
    "profile_id": profile_id,
    "num_step": 8,           # Studio Realism (8 steps, no phase noise)
    "guidance_scale": 1.3,   # Crisp Telugu plosives without metallic buzz
    "speed": 1.0,            # 1.05 for energetic YouTube hook
    "effect_preset": "raw",  # Pure acoustic vocal timbre
    "wav_bits": "16",
    "stream": "false"
}

t0 = time.time()
print(f"🚀 Generating voiceover on T4 GPU...")
res = requests.post(f"{BASE_URL}/generate", data=payload, timeout=300)
if res.status_code != 200:
    raise RuntimeError(f"Generation error {res.status_code}: {res.text}")

raw_wav_bytes = res.content
gen_duration = time.time() - t0
print(f"✅ Neural generation completed in {gen_duration:.1f} seconds!")

# ── 4. Studio Presence Air Polish & Mastering ────────────────────────────────
with wave.open(io.BytesIO(raw_wav_bytes), "rb") as wf:
    params = wf.getparams()
    audio = np.frombuffer(wf.readframes(params.nframes), dtype=np.int16).astype(np.float32) / 32768.0

# +1.8dB High-shelf presence EQ above 4.5kHz (restores natural vocal brilliance)
gain_db = 1.8
A = 10.0 ** (gain_db / 40.0)
w0 = 2.0 * np.pi * 4500.0 / params.framerate
alpha = np.sin(w0) / 2.0 * np.sqrt(2.0)
cos_w0 = np.cos(w0)

b0 = A * ((A + 1.0) + (A - 1.0) * cos_w0 + 2.0 * np.sqrt(A) * alpha)
b1 = -2.0 * A * ((A - 1.0) + (A + 1.0) * cos_w0)
b2 = A * ((A + 1.0) + (A - 1.0) * cos_w0 - 2.0 * np.sqrt(A) * alpha)
a0 = (A + 1.0) - (A - 1.0) * cos_w0 + 2.0 * np.sqrt(A) * alpha
a1 = 2.0 * ((A - 1.0) - (A + 1.0) * cos_w0)
a2 = (A + 1.0) - (A - 1.0) * cos_w0 - 2.0 * np.sqrt(A) * alpha

b = np.array([b0, b1, b2]) / a0
a = np.array([a0, a1, a2]) / a0
polished = np.clip(signal.lfilter(b, a, audio), -0.98, 0.98)

timestamp = int(time.time())
out_wav = f"/content/voiceover_{timestamp}.wav"
out_mp3 = f"/content/voiceover_{timestamp}.mp3"

with wave.open(out_wav, "wb") as out_wf:
    out_wf.setparams(params)
    out_wf.writeframes((polished * 32767.0).astype(np.int16).tobytes())

# Master to YouTube / Broadcast Standard -16 LUFS MP3
subprocess.run(
    f'ffmpeg -y -i "{out_wav}" -af loudnorm=I=-16:TP=-1.5:LRA=11 -b:a 192k "{out_mp3}" -loglevel error',
    shell=True,
    check=False,
)

audio_dur = len(audio) / params.framerate
print(f"\n🎉 Voiceover Ready! Audio Length: {audio_dur:.2f}s | Saved: {out_mp3}")

# Display Inline Audio Player and Trigger Direct Download to Laptop
if display and Audio:
    display(Audio(out_mp3))

try:
    import importlib
    colab_files = importlib.import_module("google.colab.files")
    colab_files.download(out_mp3)
    print("⬇️ Mastered MP3 downloaded directly to your laptop!")
except Exception:
    if display and HTML:
        display(HTML(f'<a href="/files{out_mp3}" download style="display:inline-block;padding:10px 18px;background:#10b981;color:#fff;border-radius:8px;font-weight:bold;text-decoration:none;">⬇️ Download Mastered Voiceover (.mp3)</a>'))

